# 06 — Hyperparameter Tuning (Random Forest)
**E-Waste Toxic Gas Detection System — ML Pipeline**

**Purpose:** Fine-tune the best model (Random Forest) using GridSearchCV with Stratified K-Fold. Justify selected hyperparameters for the research paper.

**Author:** Sanjula Madushanka | Final Year Research Y4S2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import time
import warnings
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import f1_score, accuracy_score, classification_report
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
DATA_DIR  = Path('../datasets/processed/train_test_split')
MODEL_DIR = Path('../models')
SAVE_DIR  = Path('../results')

X_train = pd.read_csv(DATA_DIR / 'X_train.csv').values
X_test  = pd.read_csv(DATA_DIR / 'X_test.csv').values
y_train = pd.read_csv(DATA_DIR / 'y_train.csv').values.ravel()
y_test  = pd.read_csv(DATA_DIR / 'y_test.csv').values.ravel()
le      = joblib.load(MODEL_DIR / 'label_encoder.pkl')

print('Data loaded ✅')
print(f'Training: {X_train.shape} | Test: {X_test.shape}')

## 6.1 Baseline Performance (Before Tuning)

In [ ]:
baseline_rf = joblib.load(MODEL_DIR / 'random_forest_v1.pkl')
y_pred_baseline = baseline_rf.predict(X_test)
baseline_acc = accuracy_score(y_test, y_pred_baseline)
baseline_f1  = f1_score(y_test, y_pred_baseline, average='weighted', zero_division=0)

print(f'BASELINE Random Forest:')
print(f'  Test Accuracy: {baseline_acc*100:.2f}%')
print(f'  F1 Weighted:   {baseline_f1*100:.2f}%')
print(f'  n_estimators:  {baseline_rf.n_estimators}')
print(f'  max_depth:     {baseline_rf.max_depth}')

## 6.2 Randomized Search (Efficient Exploration)

In [ ]:
from scipy.stats import randint

param_dist = {
    'n_estimators':     randint(100, 600),
    'max_depth':        [None, 10, 20, 30, 40],
    'min_samples_split':[2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features':     ['sqrt', 'log2', None],
    'class_weight':     [None, 'balanced'],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
base_rf = RandomForestClassifier(random_state=42, n_jobs=-1)

print('Running RandomizedSearchCV (n_iter=30, 5-fold CV)...')
print('This may take 1-3 minutes...')

t0 = time.time()
random_search = RandomizedSearchCV(
    base_rf,
    param_distributions=param_dist,
    n_iter=30,             # Test 30 random combinations
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    random_state=42,
    verbose=1
)
random_search.fit(X_train, y_train)
t1 = time.time()

print(f'\nRandomizedSearch completed in {t1-t0:.1f}s')
print(f'Best CV F1 (weighted): {random_search.best_score_*100:.2f}%')
print(f'Best params: {random_search.best_params_}')

## 6.3 Fine-Grained GridSearch Around Best Params

In [ ]:
best = random_search.best_params_

# Build narrow grid around best params
n_est = best['n_estimators']
depth = best['max_depth']

fine_grid = {
    'n_estimators':     [max(50, n_est-100), n_est, n_est+100],
    'max_depth':        [depth, None] if depth else [None, 20],
    'min_samples_split':[best['min_samples_split']],
    'min_samples_leaf': [best['min_samples_leaf']],
    'max_features':     [best['max_features']],
    'class_weight':     [best['class_weight']],
}

print('Running Fine GridSearchCV...')
t0 = time.time()
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    fine_grid,
    cv=cv,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=0
)
grid_search.fit(X_train, y_train)
t1 = time.time()

print(f'GridSearch completed in {t1-t0:.1f}s')
print(f'Best CV F1: {grid_search.best_score_*100:.2f}%')
print(f'Best params: {grid_search.best_params_}')

## 6.4 Evaluate Tuned Model vs Baseline

In [ ]:
tuned_rf = grid_search.best_estimator_
y_pred_tuned = tuned_rf.predict(X_test)
tuned_acc = accuracy_score(y_test, y_pred_tuned)
tuned_f1  = f1_score(y_test, y_pred_tuned, average='weighted', zero_division=0)

print('=' * 55)
print(f'  BEFORE TUNING vs AFTER TUNING')
print('=' * 55)
print(f'  Metric            Before      After      Δ')
print(f'  ─────────────────────────────────────────')
print(f'  Accuracy        {baseline_acc*100:>8.2f}%  {tuned_acc*100:>8.2f}%  '
      f'{(tuned_acc-baseline_acc)*100:>+6.2f}%')
print(f'  F1 (weighted)   {baseline_f1*100:>8.2f}%  {tuned_f1*100:>8.2f}%  '
      f'{(tuned_f1-baseline_f1)*100:>+6.2f}%')
print()
print('Final model hyperparameters:')
for k, v in grid_search.best_params_.items():
    print(f'  {k}: {v}')

## 6.5 Save Final Tuned Model

In [ ]:
# Save as the primary model used by FastAPI
tuned_path = MODEL_DIR / 'random_forest_v1.pkl'  # Overwrite with tuned version
joblib.dump(tuned_rf, tuned_path)
print(f'✅ Tuned Random Forest saved to: {tuned_path}')
print(f'   File size: {tuned_path.stat().st_size / 1024:.1f} KB')

print()
print('This model will be copied to backend/ml_models/ for FastAPI inference')
print()
print('=== HYPERPARAMETER JUSTIFICATION FOR PAPER ===')
for k, v in grid_search.best_params_.items():
    print(f'  {k} = {v}')
print()
print('Selected by: 5-fold stratified GridSearchCV optimising F1 weighted score')
print('Rationale: F1 weighted preferred over accuracy to account for class imbalance')
print()
print('✅ Notebook 06 complete — proceed to 07_model_export.ipynb')